# Continuous RUL Benchmark (CRB) — full pipeline on Google Colab (T4 GPU)

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** → approve the Google Drive pop-up with **any Google account**

Everything is saved in **that account's Google Drive** under `My Drive/CRB/results`:
benchmark leaderboard, every run, all studies, logs and status. Each finished run is saved
immediately — after a disconnect or runtime reset, just **Run all** again and it resumes.

With `RUN_EVERYTHING = True` the pipeline runs: N-CMAPSS download (from NASA, ~15 GB) → smoke test →
full benchmark on CMAPSS FD001–FD004, PHM08 and N-CMAPSS DS01/02/03/05/07 → all studies
(condition clustering, N-CMAPSS multi-file, SCANIA classification, LSTM ablation).

## 0. Settings

In [ ]:
REPO_URL     = "https://github.com/Nafis878/Continuous-.git"
BRANCH       = "main"
WORKDIR      = "/content/Continuous"

DRIVE_FOLDER = "CRB"     # everything goes to My Drive/CRB in the account you mount
RUN_EVERYTHING = True    # N-CMAPSS + benchmark + all studies (= run_pipeline.py --all)

# Start from results already committed in this repo: their finished benchmark runs are skipped,
# so only what is missing runs (the 2026-11-24 T4 run finished the whole benchmark).
# Set to "" to run everything from scratch.
RESUME_FROM = "results/crb_2026-11-24_T4"

# N-CMAPSS raw data (~15 GB). False: kept on the Colab disk and re-downloaded automatically in a new
# session (a few minutes). True: also kept in My Drive/CRB/N-CMAPSS (needs ~15 GB free Drive space).
KEEP_NCMAPSS_IN_DRIVE = False

SEEDS    = "42 43 44"
DATASETS = ""   # "" = all; e.g. "FD001 FD004 NCMAPSS-DS01"
# Same 13 models as the committed 2026-11-24 run (incl. RandomForest), so resuming matches it exactly
MODELS   = "MeanRUL Ridge HistGB RandomForest LSTM BiLSTM-Attn CNN1D CNN-LSTM TCN Transformer CATT CATT+TCT TCN+TCT"

## 1. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU (CPU works but is much slower).")

## 2. Get the code from GitHub

In [ ]:
import os
# Private repo? Uncomment to enter a GitHub token when the cell runs (never save it in the notebook):
# from getpass import getpass
# REPO_URL = f"https://{getpass('GitHub token: ')}@github.com/Nafis878/Continuous-.git"

os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"   # large LFS data is fetched by the pipeline only when needed
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    !git -C {WORKDIR} fetch -q origin {BRANCH} && git -C {WORKDIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1
assert os.path.isfile("run_pipeline.py"), "clone failed - check REPO_URL / access"

## 3. Mount your Google Drive

Pick any Google account in the pop-up — all results are stored in *that* Drive.

In [ ]:
import shutil
from google.colab import drive

def drive_ok():
    try:
        return os.path.ismount("/content/drive") and os.listdir("/content/drive") is not None
    except OSError:   # e.g. [Errno 107] Transport endpoint is not connected
        return False

if os.path.ismount("/content/drive") and not drive_ok():
    print("Drive connection dropped -> re-mounting ...")
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount("/content/drive", force_remount=True)
elif not os.path.ismount("/content/drive"):
    if os.path.exists("/content/drive"):
        # leftover local folder from running before Drive was mounted - NOT your Google Drive;
        # it would make drive.mount() fail with "Mountpoint must not already contain files"
        shutil.rmtree("/content/drive")
        print("Removed leftover local /content/drive folder.")
    drive.mount("/content/drive")
assert drive_ok(), "Google Drive is not mounted - re-run this cell and approve the pop-up."
BASE = os.path.join("/content/drive/MyDrive", DRIVE_FOLDER)
OUT = os.path.join(BASE, "results")
os.makedirs(OUT, exist_ok=True)
NCMAPSS_DIR = os.path.join(BASE, "N-CMAPSS") if KEEP_NCMAPSS_IN_DRIVE else "/content/N-CMAPSS"
os.makedirs(NCMAPSS_DIR, exist_ok=True)
print("Results  ->", OUT)
print("N-CMAPSS ->", NCMAPSS_DIR)
if os.path.exists(os.path.join(OUT, "benchmark", "runs.csv")):
    import pandas as pd
    _r = pd.read_csv(os.path.join(OUT, "benchmark", "runs.csv"))
    print(f"Found {int((_r['status'] == 'ok').sum())} finished benchmark runs -> resuming.")

## 4. Run the full pipeline

Environment check → N-CMAPSS download → smoke test (stops early on any error) → benchmark → studies → summary.
Output streams below; it is also logged to `My Drive/CRB/results/logs/`.

In [ ]:
args = f'--install --out "{OUT}" --ncmapss-dir "{NCMAPSS_DIR}" --seeds {SEEDS}'
if RUN_EVERYTHING:
    args += " --all"
if RESUME_FROM.strip():
    args += f' --resume-from "{RESUME_FROM}"'
if DATASETS.strip():
    args += f" --datasets {DATASETS}"
if MODELS.strip():
    args += f" --models {MODELS}"
print("python run_pipeline.py", args)
!python run_pipeline.py {args}

## 5. Leaderboard

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
lb = os.path.join(OUT, "benchmark", "LEADERBOARD.md")
if os.path.exists(lb):
    display(Markdown(open(lb, encoding="utf-8").read()))
else:
    print("No leaderboard yet - see the pipeline output / logs above.")
st = os.path.join(OUT, "pipeline_status.json")
if os.path.exists(st):
    import json
    for s in json.load(open(st))["stages"]:
        print(f"{s['stage']:20s} {s['status']}")

## 6. (Optional) download a zip of the results

Results are already in your Drive; this is only for a local copy.

In [ ]:
import shutil, time
archive = shutil.make_archive(f"/content/crb-results-{time.strftime('%Y%m%d-%H%M%S')}", "zip", OUT)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("Download via the Files panel instead:", exc)